# Tutorial 9: Capstone — Reproducible Scientific Pipeline

Estimated time: 45-60 minutes

## Prerequisites
All previous tutorials' deps. Run `mm doctor`.

## Learning aims
- Run end-to-end: data -> surrogate -> metamodel sample -> reproducible artifact
- Capture seeds, spec digests, dependency versions for a citable run


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Self-contained: walks up to find the repo root, adds src/ to sys.path,
# then imports the official bootstrap helper for chdir + later run_cli use.
import os, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    _marker = _candidate / 'pyproject.toml'
    if _marker.is_file() and 'name = "metamodeler"' in _marker.read_text():
        ROOT = _candidate
        break
else:
    raise FileNotFoundError('Could not locate metamodeler repo root from ' + str(_here))

_src = str((ROOT / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
if Path.cwd().resolve() != ROOT.resolve():
    os.chdir(ROOT)

from metamodeler.tutorial import bootstrap, run_cli  # noqa: E402
ROOT = bootstrap()
print('Repo root:', ROOT)


## Step 1: Generate runs

In [ ]:
run_cli('run', 'tutorials/specs/model.toy.grid.json')


## Step 2: Fit a surrogate (PyMC if available, else linear fallback)

In [ ]:
from metamodeler.config import diagnose
report = diagnose()
have_pymc = report['backends']['pymc']['installed']
spec_path = (
    'tutorials/specs/surrogate.toy.pymc_gp.json'
    if have_pymc else 'tutorials/specs/surrogate.toy.pymc_gp.json'
)
run_cli('surrogate', 'fit', spec_path)


## Step 3: Build & sample the metamodel

In [ ]:
run_cli('meta', 'build', 'examples/metamodels/metamodel.simple.json')
run_cli('meta', 'sample', 'examples/metamodels/metamodel.simple.json',
        '--draws', '200', '--tune', '100', '--chains', '2', '--seed', '7')


## Step 4: Print provenance

In [ ]:
import json

reg = ROOT / 'tmp/metamodel_samples_registry.json'
if reg.is_file():
    payload = json.loads(reg.read_text())
    latest_id = sorted(payload.keys())[-1]
    print(json.dumps(payload[latest_id], indent=2, sort_keys=True))
else:
    print('No metamodel samples registry found yet.')


## Capstone reflection
- Which artifacts together fully reproduce this run? Spec digest, seed, dataset digest, dependency versions.
- Which step would benefit most from improved uncertainty quantification?
